# SyN image registration

Register ANTsPy's 2-D r64 sample to r16 with ANTsTorch.
**Resources:** downloads the two sample images on first use; CPU by default.
The short iteration schedule demonstrates the API and needs tuning for real data.

Install ANTsTorch and notebook tools from the repository root before opening this notebook:
```bash
python -m pip install -e '.[notebooks]'
python -m ipykernel install --user --name python3 --display-name "Python 3 (ANTsTorch)"
```
Select that kernel and run the cells in order. All examples default to CPU.

In [ ]:
import torch  # Import before ANTs to avoid OpenMP conflicts on macOS.
import numpy as np
import ants
import antstorch

# Display figures inside Jupyter/VS Code even if MPLBACKEND was set to Agg.
from IPython import get_ipython
shell = get_ipython()
if shell is not None:
    shell.run_line_magic("matplotlib", "inline")
import matplotlib.pyplot as plt

device = "cpu"
antstorch.set_default_device(device)

In [ ]:
from pathlib import Path
from antstorch.syn import syn_registration

output_dir = Path("example_outputs/syn")
output_dir.mkdir(parents=True, exist_ok=True)
fixed = ants.image_read(ants.get_ants_data("r16")).clone("float")
moving = ants.image_read(ants.get_ants_data("r64")).clone("float")
result = syn_registration(
    fixed, moving, type_of_transform="SyN", device=device,
    levels=(4, 2, 1), reg_iterations=(20, 10, 5),
    outprefix=str(output_dir / "r64_to_r16_"), verbose=True,
)

## Inspect alignment and deformation
The warped moving image and Jacobian should be finite and use the fixed grid.
These structural checks do not replace visual inspection or registration validation.

In [ ]:
warped = result["warpedmovout"]
jacobian = result["jacobian"]
for output in (warped, jacobian):
    assert output.shape == fixed.shape
    assert np.isfinite(output.numpy()).all()
    np.testing.assert_allclose(output.spacing, fixed.spacing)
    np.testing.assert_allclose(output.origin, fixed.origin)
    np.testing.assert_allclose(output.direction, fixed.direction)
fig, axes = plt.subplots(1, 4, figsize=(12, 3), constrained_layout=True)
for ax, image, title in zip(axes, [fixed, moving, warped, jacobian],
                            ["Fixed r16", "Moving r64", "Warped r64", "Jacobian"]):
    ax.imshow(image.numpy().T, origin="lower", cmap="gray")
    ax.set_title(title); ax.axis("off")
plt.show()
ants.image_write(warped, str(output_dir / "warped.nii.gz"))
ants.image_write(jacobian, str(output_dir / "jacobian.nii.gz"))
print("Forward transforms:", result["fwdtransforms"])
print("Outputs:", output_dir.resolve())

Outputs are relative to the kernel working directory; rerunning overwrites matching files.